# ViCAM

In [1]:
import os
import torch
from esm.models.esmc import ESMC
from esm.tokenization import get_esmc_model_tokenizers
tokenizer = get_esmc_model_tokenizers()

/stor/work/Wilke/luiz/ViCAM/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
padding_token=tokenizer.pad_token_id
mask_token=tokenizer.mask_token_id
no_mask_tokens=[tokenizer.cls_token_id, tokenizer.eos_token_id]
n_tokens=tokenizer.vocab_size

print(f"Padding token: {padding_token}, Mask token: {mask_token}, No mask tokens: {no_mask_tokens}, Vocabulary size: {n_tokens}")

Padding token: 1, Mask token: 32, No mask tokens: [0, 2], Vocabulary size: 33


In [3]:
# load the models locally
def ESMC_300M_202412(model_path: str, device: torch.device | str = "cpu"):
    with torch.device(device):
        model = ESMC(
            d_model=960, n_heads=15, n_layers=30, tokenizer=get_esmc_model_tokenizers()
        )
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    # Convert model parameters to torch.bfloat16 or torch.float32
    model = model.to(torch.float32)
    return model


def ESMC_600M_202412(model_path: str, device: torch.device | str = "cpu"):
    with torch.device(device):
        model = ESMC(
            d_model=1152, n_heads=18, n_layers=36, tokenizer=get_esmc_model_tokenizers()
        )
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    # Convert model parameters to float32
    model = model.to(torch.float32)
    return model

In [4]:
checkpoint_path = '../checkpoints/ESMC/esmc_300m_2024_12_v0.pth'

model = ESMC_300M_202412(checkpoint_path)
model

ESMC(
  (embed): Embedding(64, 960)
  (transformer): TransformerStack(
    (blocks): ModuleList(
      (0-29): 30 x UnifiedTransformerBlock(
        (attn): MultiHeadAttention(
          (layernorm_qkv): Sequential(
            (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=960, out_features=2880, bias=False)
          )
          (out_proj): Linear(in_features=960, out_features=960, bias=False)
          (q_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (k_ln): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (rotary): RotaryEmbedding()
        )
        (ffn): Sequential(
          (0): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=960, out_features=5120, bias=False)
          (2): SwiGLU()
          (3): Linear(in_features=2560, out_features=960, bias=False)
        )
      )
    )
    (norm): LayerNorm((960,), eps=1e-05, elementwise_affine=True)
  )
  (sequ

In [16]:
tokenizer

EsmSequenceTokenizer(name_or_path='', vocab_size=33, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<cls>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'cls_token': '<cls>', 'mask_token': '<mask>', 'additional_special_tokens': ['|']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<cls>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<eos>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	31: AddedToken("|", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32: AddedToken("<mask>", rstrip=False, lstrip=False, single_word=False, 

In [370]:
inputs = tokenizer(['SNEWHYAYIQWHVDLMKLLHNYMCYFNHLR<mask>DPHPRGCQMSEGCKKIVDHCKWAEQHEEPKKIKE<mask>CEQDEMNECQGSKQQVCLVTKCIWETAIDDTACWRFDINMLG', 
                     'SNEWHYAYIQWHVDLMKLLHYMCYFNHLRQDPHPRGCQMSEGCKKIVDHCKWAE<mask>HEEPKKIKEECEQDEMNEC<mask>GSKQQVCLVTKCEIWETAIDDTACWRFDINMLG'], return_tensors='pt', padding=True)
inputs

{'input_ids': tensor([[ 0,  8, 17,  9, 22, 21, 19,  5, 19, 12, 16, 22, 21,  7, 13,  4, 20, 15,
          4,  4, 21, 17, 19, 20, 23, 19, 18, 17, 21,  4, 10, 32, 13, 14, 21, 14,
         10,  6, 23, 16, 20,  8,  9,  6, 23, 15, 15, 12,  7, 13, 21, 23, 15, 22,
          5,  9, 16, 21,  9,  9, 14, 15, 15, 12, 15,  9, 32, 23,  9, 16, 13,  9,
         20, 17,  9, 23, 16,  6,  8, 15, 16, 16,  7, 23,  4,  7, 11, 15, 23, 12,
         22,  9, 11,  5, 12, 13, 13, 11,  5, 23, 22, 10, 18, 13, 12, 17, 20,  4,
          6,  2],
        [ 0,  8, 17,  9, 22, 21, 19,  5, 19, 12, 16, 22, 21,  7, 13,  4, 20, 15,
          4,  4, 21, 19, 20, 23, 19, 18, 17, 21,  4, 10, 16, 13, 14, 21, 14, 10,
          6, 23, 16, 20,  8,  9,  6, 23, 15, 15, 12,  7, 13, 21, 23, 15, 22,  5,
          9, 32, 21,  9,  9, 14, 15, 15, 12, 15,  9,  9, 23,  9, 16, 13,  9, 20,
         17,  9, 23, 32,  6,  8, 15, 16, 16,  7, 23,  4,  7, 11, 15, 23,  9, 12,
         22,  9, 11,  5, 12, 13, 13, 11,  5, 23, 22, 10, 18, 13, 12, 17, 20, 

In [6]:
outputs = model(inputs['input_ids'])
logits, embeddings, hidden_states = outputs.sequence_logits, outputs.embeddings, outputs.hidden_states

In [7]:
logits.shape, embeddings.shape, hidden_states.shape

(torch.Size([2, 110, 64]),
 torch.Size([2, 110, 960]),
 torch.Size([30, 2, 110, 960]))

In [8]:
preds = logits.view(-1, logits.shape[-1])

In [368]:
targets = tokenizer(['SNEWHYAYIQWHVDLMKLLHNYMCYFNHLRQDPHPRGCQMSEGCKKIVDHCKWAEQHEEPKKIKEECEQDEMNECQGSKQQVCLVTKCIWETAIDDTACWRFDINMLG', 
                     'SNEWHYAYIQWHVDLMKLLHYMCYFNHLRQDPHPRGCQMSEGCKKIVDHCKWAEQHEEPKKIKEECEQDEMNECQGSKQQVCLVTKCEIWETAIDDTACWRFDINMLG'],
                    return_tensors='pt', padding=True)
targets = targets['input_ids'].view(-1)
targets

tensor([ 0,  8, 17,  9, 22, 21, 19,  5, 19, 12, 16, 22, 21,  7, 13,  4, 20, 15,
         4,  4, 21, 17, 19, 20, 23, 19, 18, 17, 21,  4, 10, 16, 13, 14, 21, 14,
        10,  6, 23, 16, 20,  8,  9,  6, 23, 15, 15, 12,  7, 13, 21, 23, 15, 22,
         5,  9, 16, 21,  9,  9, 14, 15, 15, 12, 15,  9,  9, 23,  9, 16, 13,  9,
        20, 17,  9, 23, 16,  6,  8, 15, 16, 16,  7, 23,  4,  7, 11, 15, 23, 12,
        22,  9, 11,  5, 12, 13, 13, 11,  5, 23, 22, 10, 18, 13, 12, 17, 20,  4,
         6,  2,  0,  8, 17,  9, 22, 21, 19,  5, 19, 12, 16, 22, 21,  7, 13,  4,
        20, 15,  4,  4, 21, 19, 20, 23, 19, 18, 17, 21,  4, 10, 16, 13, 14, 21,
        14, 10,  6, 23, 16, 20,  8,  9,  6, 23, 15, 15, 12,  7, 13, 21, 23, 15,
        22,  5,  9, 16, 21,  9,  9, 14, 15, 15, 12, 15,  9,  9, 23,  9, 16, 13,
         9, 20, 17,  9, 23, 16,  6,  8, 15, 16, 16,  7, 23,  4,  7, 11, 15, 23,
         9, 12, 22,  9, 11,  5, 12, 13, 13, 11,  5, 23, 22, 10, 18, 13, 12, 17,
        20,  4,  6,  2])

In [10]:
from torch.nn import functional as F
loss = F.cross_entropy(preds, targets, ignore_index=tokenizer.pad_token_id)
loss

tensor(3.8862, grad_fn=<NllLossBackward0>)

## Masking

In [15]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.5, return_tensors="pt")
collated = data_collator(inputs['input_ids'])
x = collated["input_ids"]
y = collated["labels"]

In [17]:
x[0]


tensor([ 0, 32, 23,  9, 22, 21, 19, 32, 19, 12, 16, 22, 32,  7, 32, 32, 32, 15,
        32, 32, 21, 32, 19, 20, 23, 19, 32, 17, 21,  4, 10, 32, 13, 14, 21, 32,
        32,  6, 32, 16, 32,  8, 32, 32, 32, 32, 15, 32, 32, 13, 32, 23, 32, 22,
        32, 32, 16, 32, 32, 32, 32, 32, 32, 12, 15,  9, 32, 32,  9, 16,  1, 32,
        20, 32,  9, 23, 16,  6,  8, 32, 32, 16, 32, 32, 32,  7, 11, 15, 23, 12,
        32, 16, 32,  5, 32, 13, 32, 32, 32, 32, 22, 10, 18, 13, 32, 17, 20, 32,
        32,  2])

In [18]:
y[0]

tensor([-100,    8,   17, -100, -100,   21, -100,    5, -100, -100, -100, -100,
          21, -100,   13,    4,   20, -100,    4,    4, -100,   17, -100, -100,
        -100, -100,   18, -100, -100, -100, -100, -100,   13, -100, -100,   14,
          10,    6,   23, -100,   20, -100,    9,    6,   23,   15,   15,   12,
           7, -100,   21, -100,   15, -100,    5,    9, -100,   21,    9,    9,
          14,   15,   15, -100, -100, -100, -100,   23, -100, -100,   13,    9,
        -100,   17, -100, -100, -100, -100, -100,   15,   16, -100,    7,   23,
           4, -100, -100,   15, -100, -100,   22,    9,   11, -100,   12, -100,
          13,   11,    5,   23, -100, -100, -100, -100,   12, -100,   20,    4,
           6, -100])

In [365]:
from typing import List
import torch

class MLM:
    """
    ## Masked LM (MLM)

    This class implements the masking procedure for a given batch of token sequences.
    """

    def __init__(self, *,
                 padding_token: int, mask_token: int, no_mask_tokens: List[int], n_tokens: int,
                 masking_prob: float = 0.15, randomize_prob: float = 0.1, no_change_prob: float = 0.1,
                 ):
        """
        * `padding_token` is the padding token `[PAD]`.
          We will use this to mark the labels that shouldn't be used for loss calculation.
        * `mask_token` is the masking token `[MASK]`.
        * `no_mask_tokens` is a list of tokens that should not be masked.
        This is useful if we are training the MLM with another task like classification at the same time,
        and we have tokens such as `[CLS]` that shouldn't be masked.
        * `n_tokens` total number of tokens (used for generating random tokens)
        * `masking_prob` is the masking probability
        * `randomize_prob` is the probability of replacing with a random token
        * `no_change_prob` is the probability of replacing with original token
        """
        self.n_tokens = n_tokens
        self.no_change_prob = no_change_prob
        self.randomize_prob = randomize_prob
        self.masking_prob = masking_prob
        self.no_mask_tokens = no_mask_tokens + [padding_token, mask_token]
        self.padding_token = padding_token
        self.mask_token = mask_token

    def __call__(self, x: torch.Tensor):
        """
        * `x` is the batch of input token sequences.
         It's a tensor of type `long` with shape `[seq_len, batch_size]`.
        """

        # Mask `masking_prob` of tokens
        full_mask = torch.rand(x.shape, device=x.device) < self.masking_prob
        # Unmask `no_mask_tokens`
        for t in self.no_mask_tokens:
            full_mask &= x != t

        # A mask for tokens to be replaced with original tokens
        unchanged = full_mask & (torch.rand(x.shape, device=x.device) < self.no_change_prob)
        # A mask for tokens to be replaced with a random token
        random_token_mask = full_mask & (torch.rand(x.shape, device=x.device) < self.randomize_prob)
        # Indexes of tokens to be replaced with random tokens
        random_token_idx = torch.nonzero(random_token_mask, as_tuple=True)
        # Random tokens for each of the locations
        random_tokens = torch.randint(0, self.n_tokens, (len(random_token_idx[0]),), device=x.device)
        # The final set of tokens that are going to be replaced by `[MASK]`
        mask = full_mask & ~random_token_mask & ~unchanged

        # Make a clone of the input for the labels
        y = x.clone()

        # Replace with `[MASK]` tokens;
        # note that this doesn't include the tokens that will have the original token unchanged and
        # those that get replace with a random token.
        x.masked_fill_(mask, self.mask_token)
        # Assign random tokens
        x[random_token_idx] = random_tokens

        # Assign token `[PAD]` to all the other locations in the labels.
        # The labels equal to `[PAD]` will not be used in the loss.
        y.masked_fill_(~full_mask, self.padding_token)

        # Return the masked input and the labels
        return x, y


In [432]:
inputs = tokenizer(['A'*100, 'M'*100,], return_tensors='pt', padding=True)
inputs

{'input_ids': tensor([[ 0,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  2],
        [ 0, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,  2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [433]:
x = inputs['input_ids']

In [434]:
masking_prob = 0.15
full_mask = torch.rand(x.shape, device=x.device) < masking_prob
full_mask

tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
          True, False, False, False,  True, False,  True, False, False,  True,
         False,  True, False, False,  True,  True,  True, False,  True, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False,  True, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False,  True,
         False, False, False, False, False, False, False, False, False, False,
         False, False],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False,  True, False,  True, False, False, False, False,
         False, False,  True

In [438]:
full_mask[0].sum().item()

11

In [443]:
# x is your input tensor
num_tokens = x.numel()
num_mask = int(masking_prob * num_tokens)

# Generate flat indices and randomly permute them
flat_indices = torch.randperm(num_tokens, device=x.device)[:num_mask]

# Create flat mask
flat_mask = torch.zeros(num_tokens, dtype=torch.bool, device=x.device)
flat_mask[flat_indices] = True

# Reshape to match x
full_mask = flat_mask.view(x.shape)

full_mask#.sum().item()

tensor([[False, False,  True, False, False, False, False, False, False,  True,
         False,  True, False, False,  True, False, False, False, False, False,
         False,  True, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False,  True, False, False, False,  True, False, False, False, False,
         False,  True, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False,  True, False, False, False, False, False,
          True, False, False, False,  True, False, False, False, False, False,
         False,  True, False, False, False,  True, False, False, False, False,
         False, False],
        [False, False, False, False,  True, False,  True, False, False, False,
         False, False, False, False, False,  True, False, False, False, False,
         False, False, False

In [ ]:
MLM = MLM(
    padding_token=tokenizer.pad_token_id,
    mask_token=tokenizer.mask_token_id,
    no_mask_tokens=[tokenizer.cls_token_id, tokenizer.eos_token_id],
    n_tokens=tokenizer.vocab_size,
    masking_prob=0.15,
    randomize_prob=0.1,
    no_change_prob=0.1
)

In [119]:
seqs = [
    'PMHLWYWAESNQVHCWELLMCEPTGVAHSFITYNTPYDQCSADYQDPMTNEMNDCCTKVGWHGFMNPADLLITDEANVKREHVIFQEMVCCLCFFRSTYNKMFQHTHSWHNTGPNIDRTDDHALGKTVVIGWKFWMPWKSHGCTCSRQDFYNGAPVVIKLKWHFEKCHEMQFPHSVYTSKMYVREFLVAMNHSHLMFEYWDPNRWDEHLHFTSAMKRTANRVQSVKTDGIDWQIFMVSEASACSSIIHCLVEQWAYENAQQGHYCMRAGRLFVIELYQYHLVLFLWSANRYFTSQSPGNCQKQEKYPAYWKRKWICNMQCCQTRFSIFFMDTKAAGNLTFGFCNHHENTFFHNVAQDFRMNKTTNAWPGWSNPHIWAIETSYWVTMKYAEDYQAVIETGSIITMDIITCVFDCPIDIDCILCTM',
    'QGGFIAADQYYRFPAWGKAWRGPYCMCNMQAWFLENNNDRVGPECACNQWADHCLSAHKEPDYRDGAVTGRSSSMTKEVMPFTYHMFHDALFQKDCEKNTIMNYRRGDQLNDVGWRLTWIMAKPNMQTAFMESQRTLCCDILAKEVLNKTMRFICKNCKSQGYPNHLDRQPELWYMHWDVCWLLEMMYTIDNVKDLICMMVDANTWTSKHAMHPRELNMWPEFEMNCKEEARDDHFWPSHPRTVYFHCCWMMPFELYP',
    'HQHPRTYVPTMTIYLFWEEEFTPQKWRNGEYATEEQMQRMFSTQNPVCFMKTGNQYGWVCHLVPSVDMEGVCQFDRNQMNYWPSIIIMLRISCVELQPRRWYAFFCGELEGYICEPILMHQWCVVRMHKHCHGYCRQFHLDCNRFNVLRDLVTCIKIDEGFWMETFEWQMVD',
    'LKCPHVKACLKFAFGKIDEAANINYVEGSFPELLQFNQFMMAFPQLRSHHPQPACLCSQPQVHPPDRYCNTLERVSRSQCRHHRRCDTNFLLTNQQSWVPILRWMFLISTISLKPFPFFRISDDPDHYMGKKGGITYVSPGHFQYTHTIFEWCKLMMQFSPRSYAFYDHNFGHCLETHWRPLIPYHYCLPASMGIGVKALNRWGSHHKHSIKHHYIWEIRKMAHTMLWWVMRWKDTPCRPHTETGYIIPNFCEDICKQQPKLWQCFMQQVINSPGHDCDNCQGFSFVSELPDDWARAWNFRAYGLWVLCFGLSVCMFEELCHKSTNAMERSGAEFNWWQSRMKSTVVY',
    'KRSLHMNYRGMARSVFQWVDYPLMSKWMVTYGAFTAYLEKMYWAGYCEDHGVMYFEEGEYWKWIYELGADGKEQHVPGGVQNPNDGGRRLIFAGFDDGETNLNMPDHLMPWMSHFMDNHMGNLHEYCARPCGVFNDCQFKLCILPHLCRSNTGKKEQWCDWVKIDPCATWYDEETKSQYIQTQCYCSKNNMEKRSVTPPPANNQLSSINWVWRWVLAIHPQLCSKEWTNPESHSPPQILGPDPFWYMRDVLTLAFLKTEKVPWGECPRCKFWTAIHRRIDLNVPNGEHYIKAYPWEAMNNEDLDMTDQWAQDWYHQQLCMEGSEMDLHKTLRMSFNTAFRMWHDATMPISFDGGQCWVPMAPMFHYRWFLWTGFHFFAIGSLAYFVFIHSNRI',
    'YMPETWEGEQQMYVDVSNQCAWGRWEKVWESCRGPDYYTRSIWCPTVFNLRWLEWPVEMNVHEEFNMMIPQTVGAPYASVHRCVWTMFKTQWGWKMPVSSYQKKVLMVSFGAGPFPKQTVDSSTHVWNHIECQAHSFKQAYHLGNDDLPCDRIQKVSQYAKVHLREGFFDKKVLQTNVGVVKQRPLETFCQFAEQLSRPCSPKWQVTQLPARAFHDLDLVSRCWVHWFVFYMCDECWMRRNPMDKWVKMSTRYNFRNDCNWHVWQFAIKKQFYDQWPIKIPGAMNLIMVMILIPHGKIPNHDWCRMNGKRGQFMPQVCCVCGPLVWYDDTLNLSHLEGVLCVVLEYLLRCMCLDNESFDDCRWGAPHGRICNRGPYDRNCWPLKHIEWSHCSTFITNVMRITFADCRYIPIHKRDFPGKYVQGSKLLTRWCYQFPHCDDAMIPQGFYPALWAW',
    'SNEWHYAYIQWHVDLMKLLHNYMCYFNHLRQDPHPRGCQMSEGCKKIVDHCKWAEQHEEPKKIKEECEQDEMNECQGSKQQVCLVTKCEIWETAIDDTACWRFDINMLG',
]

seqs = [
    'HQHPRTYVPTMTIYLFWEEEFTPQKWRNGEYATEEQMQRMFSTQNPVCFMKTGNQYGWVCHLVPSVDMEGVCQFDRNQMNYWPSIIIMLRISCVELQPRRWYAFFCGELEGYICEPILMHQWCVVRMHKHCHGYCRQFHLDCNRFNVLRDLVTCIKIDEGFWMETFEWQMVD',
]


In [120]:
tokens = tokenizer(seqs, padding=True, return_tensors='pt')
tokens = tokens['input_ids']

In [122]:
x, y = MLM(tokens)

In [123]:
x[0]

tensor([ 0, 21, 32, 21, 14, 10, 11, 19,  7, 14, 11, 20, 11, 12, 19,  4, 18, 22,
         9,  9,  9, 18, 32, 32, 16, 15, 32, 10, 17,  6,  9, 19,  5, 11,  9,  9,
        16, 20, 16, 32, 20, 18,  8, 11, 16, 17, 14,  7, 23, 18, 20, 15, 11,  6,
        17, 16, 19,  6, 22,  7, 32, 21,  4,  7, 14,  8, 32, 13, 20,  9,  6,  7,
        23, 16, 18, 13, 10, 17, 16, 20, 17, 19, 22, 14,  8, 12, 12, 12, 20,  4,
        10, 12,  8, 23, 32,  9,  4, 32, 14, 32, 10, 22, 19,  5, 18, 18, 23,  6,
        25,  4,  9,  6, 32, 12, 23,  9, 14, 12,  4, 20, 21, 16, 22, 23,  7, 32,
        10, 20, 21, 15, 21, 23, 21, 32, 19, 23, 10, 16, 18, 21,  4, 13, 23, 17,
        10, 18, 17,  7,  4, 10, 13,  4,  7, 11, 23, 32, 15, 12, 13,  9, 32, 18,
        22, 20,  9, 11, 18,  9, 22, 16, 20,  7, 13,  2])

In [124]:
y[0]

tensor([ 1,  1, 16,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1, 11, 14,  1,  1, 22,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1, 10,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1, 23,  1,  1,  1,  1,  1,  7, 13,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  7,  1,  1, 16,  1, 10,  1,  1,  1,  1,  1,  1,  1,  1,
         9,  1,  1,  1, 19,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  7,
         1,  1,  1,  1,  1,  1,  1,  6,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, 12,  1,  1,  1,  1,  6,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1])

In [125]:
tokens[0]

tensor([ 0, 21, 32, 21, 14, 10, 11, 19,  7, 14, 11, 20, 11, 12, 19,  4, 18, 22,
         9,  9,  9, 18, 32, 32, 16, 15, 32, 10, 17,  6,  9, 19,  5, 11,  9,  9,
        16, 20, 16, 32, 20, 18,  8, 11, 16, 17, 14,  7, 23, 18, 20, 15, 11,  6,
        17, 16, 19,  6, 22,  7, 32, 21,  4,  7, 14,  8, 32, 13, 20,  9,  6,  7,
        23, 16, 18, 13, 10, 17, 16, 20, 17, 19, 22, 14,  8, 12, 12, 12, 20,  4,
        10, 12,  8, 23, 32,  9,  4, 32, 14, 32, 10, 22, 19,  5, 18, 18, 23,  6,
        25,  4,  9,  6, 32, 12, 23,  9, 14, 12,  4, 20, 21, 16, 22, 23,  7, 32,
        10, 20, 21, 15, 21, 23, 21, 32, 19, 23, 10, 16, 18, 21,  4, 13, 23, 17,
        10, 18, 17,  7,  4, 10, 13,  4,  7, 11, 23, 32, 15, 12, 13,  9, 32, 18,
        22, 20,  9, 11, 18,  9, 22, 16, 20,  7, 13,  2])

In [19]:
x.shape, y.shape

(torch.Size([7, 455]), torch.Size([7, 455]))

In [134]:
logits = model(x).sequence_logits
logits.shape

torch.Size([7, 455, 64])

In [138]:
preds = logits.view(-1, logits.shape[-1])
preds.shape

torch.Size([3185, 64])

In [137]:
targets = y.view(-1)
targets.shape

torch.Size([3185])

In [139]:
loss = F.cross_entropy(preds, targets, ignore_index=tokenizer.pad_token_id)
loss

tensor(3.0707, grad_fn=<NllLossBackward0>)

# Creating a data splits

In [151]:
from Bio import SeqIO

input_fasta  = '../data/raw/URVDBv29-prot_clustered.fasta'
records = list(SeqIO.parse(input_fasta, "fasta"))

In [152]:
lengths = [len(record.seq) for record in records]

In [157]:
import numpy as np
lengths = np.array(lengths)

In [159]:
lengths.max(), lengths.min(), lengths.mean(), 

(np.int64(13556), np.int64(11), np.float64(366.8360336687927))

In [ ]:
lengths = [len(record.seq) for record in records]
for record in records:
    print(record.id)
    print(record.seq)
    print(record.description)
    print(record.format("fasta"))
    break

# ESM2 MLM

In [1]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM, EsmModel

tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
model = EsmForMaskedLM.from_pretrained("facebook/esm2_t33_650M_UR50D")
model.train()

/stor/work/Wilke/luiz/ViCAM/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 1280, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05, 

In [23]:
del model.esm.contact_head

model

EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 1280, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05, 

In [2]:
padding_token=tokenizer.pad_token_id
mask_token=tokenizer.mask_token_id
no_mask_tokens=[tokenizer.cls_token_id, tokenizer.eos_token_id]
n_tokens=tokenizer.vocab_size
print(f"Padding token: {padding_token}, Mask token: {mask_token}, No mask tokens: {no_mask_tokens}, Vocabulary size: {n_tokens}")

Padding token: 1, Mask token: 32, No mask tokens: [0, 2], Vocabulary size: 33


In [3]:
seqs = [
    'A'*100,
    'QGGFIAADQYYRFPAWGKAWRGPYCMCNMQAWFLENNNDRVGPECACNQWADHCLSAHKEPDYRDGAVTGRSSSMTKEVMPFTYHMFHDALFQKDCEKNTIMNYRRGDQLNDVGWRLTWIMAKPNMQTAFMESQRTLCCDILAKEVLNKTMRFICKNCKSQGYPNHLDRQPELWYMHWDVCWLLEMMYTIDNVKDLICMMVDANTWTSKHAMHPRELNMWPEFEMNCKEEARDDHFWPSHPRTVYFHCCWMMPFELYP',
    'HQHPRTYVPTMTIYLFWEEEFTPQKWRNGEYATEEQMQRMFSTQNPVCFMKTGNQYGWVCHLVPSVDMEGVCQFDRNQMNYWPSIIIMLRISCVELQPRRWYAFFCGELEGYICEPILMHQWCVVRMHKHCHGYCRQFHLDCNRFNVLRDLVTCIKIDEGFWMETFEWQMVD',
    'LKCPHVKACLKFAFGKIDEAANINYVEGSFPELLQFNQFMMAFPQLRSHHPQPACLCSQPQVHPPDRYCNTLERVSRSQCRHHRRCDTNFLLTNQQSWVPILRWMFLISTISLKPFPFFRISDDPDHYMGKKGGITYVSPGHFQYTHTIFEWCKLMMQFSPRSYAFYDHNFGHCLETHWRPLIPYHYCLPASMGIGVKALNRWGSHHKHSIKHHYIWEIRKMAHTMLWWVMRWKDTPCRPHTETGYIIPNFCEDICKQQPKLWQCFMQQVINSPGHDCDNCQGFSFVSELPDDWARAWNFRAYGLWVLCFGLSVCMFEELCHKSTNAMERSGAEFNWWQSRMKSTVVY',
    'KRSLHMNYRGMARSVFQWVDYPLMSKWMVTYGAFTAYLEKMYWAGYCEDHGVMYFEEGEYWKWIYELGADGKEQHVPGGVQNPNDGGRRLIFAGFDDGETNLNMPDHLMPWMSHFMDNHMGNLHEYCARPCGVFNDCQFKLCILPHLCRSNTGKKEQWCDWVKIDPCATWYDEETKSQYIQTQCYCSKNNMEKRSVTPPPANNQLSSINWVWRWVLAIHPQLCSKEWTNPESHSPPQILGPDPFWYMRDVLTLAFLKTEKVPWGECPRCKFWTAIHRRIDLNVPNGEHYIKAYPWEAMNNEDLDMTDQWAQDWYHQQLCMEGSEMDLHKTLRMSFNTAFRMWHDATMPISFDGGQCWVPMAPMFHYRWFLWTGFHFFAIGSLAYFVFIHSNRI',
    'YMPETWEGEQQMYVDVSNQCAWGRWEKVWESCRGPDYYTRSIWCPTVFNLRWLEWPVEMNVHEEFNMMIPQTVGAPYASVHRCVWTMFKTQWGWKMPVSSYQKKVLMVSFGAGPFPKQTVDSSTHVWNHIECQAHSFKQAYHLGNDDLPCDRIQKVSQYAKVHLREGFFDKKVLQTNVGVVKQRPLETFCQFAEQLSRPCSPKWQVTQLPARAFHDLDLVSRCWVHWFVFYMCDECWMRRNPMDKWVKMSTRYNFRNDCNWHVWQFAIKKQFYDQWPIKIPGAMNLIMVMILIPHGKIPNHDWCRMNGKRGQFMPQVCCVCGPLVWYDDTLNLSHLEGVLCVVLEYLLRCMCLDNESFDDCRWGAPHGRICNRGPYDRNCWPLKHIEWSHCSTFITNVMRITFADCRYIPIHKRDFPGKYVQGSKLLTRWCYQFPHCDDAMIPQGFYPALWAW',
    'SNEWHYAYIQWHVDLMKLLHNYMCYFNHLRQDPHPRGCQMSEGCKKIVDHCKWAEQHEEPKKIKEECEQDEMNECQGSKQQVCLVTKCEIWETAIDDTACWRFDINMLG',
]

seqs = ['A'*100,]
seqs

['AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA']

In [4]:
inputs = tokenizer(seqs, padding=True, return_tensors="pt")

In [5]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15, return_tensors="pt")
collated = data_collator(inputs["input_ids"])
x = collated["input_ids"]
y = collated["labels"]
attention_mask = inputs["attention_mask"]

In [6]:
x[0]

tensor([ 0,  0,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
         5,  1,  5, 32,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
        32,  5,  5,  5, 32,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  8,
         5,  5, 32,  5, 32,  5,  5,  5,  5,  5,  5, 32,  5, 12,  5,  5,  5, 32,
        32, 32,  5,  5,  5,  5,  5, 32,  5, 32,  5,  5,  5,  5, 32,  5,  5, 32,
         5,  5, 32,  5,  5, 32, 32,  5,  5, 32,  5,  2])

In [7]:
count=0
for i in x[0]:
    #if i not in [0, 2, 5]:
    if i !=5:
        count +=1

print(count/len(x[0]))

0.22549019607843138


In [8]:
x[0].shape 

torch.Size([102])

In [9]:
y[0].shape 

torch.Size([102])

In [10]:
attention_mask[0].shape 

torch.Size([102])

In [11]:
outputs = model(input_ids=x, attention_mask=attention_mask, labels=y, output_hidden_states=True)
outputs

MaskedLMOutput(loss=tensor(0.0582, grad_fn=<NllLossBackward0>), logits=tensor([[[ 15.1637,  -8.1877,   2.8143,  ..., -14.0755, -14.4777,  -8.2164],
         [ -7.4765, -16.4134,  -8.0899,  ..., -15.3485, -14.8985, -16.3759],
         [-10.7473, -15.0648,  -9.3695,  ..., -14.5629, -15.0044, -15.0701],
         ...,
         [-11.0623, -18.4554, -10.1851,  ..., -14.7522, -14.5882, -18.4171],
         [-11.6858, -15.5456,  -9.8657,  ..., -14.4394, -14.7645, -15.5139],
         [  0.9148,  -9.3476,  19.8629,  ..., -14.2594, -13.7868,  -9.3326]]],
       grad_fn=<AddBackward0>), hidden_states=(tensor([[[ 0.0566, -0.0615, -0.1373,  ..., -0.2344,  0.1247, -0.0697],
         [ 0.0566, -0.0615, -0.1373,  ..., -0.2344,  0.1247, -0.0697],
         [-0.0420, -0.0421, -0.0509,  ..., -0.0582, -0.2253, -0.0768],
         ...,
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [-0.0420, -0.0421, -0.0509,  ..., -0.0582, -0.2253, -0.0768],
         [-0.0946, -0.0473, -0.0607

In [12]:
outputs.loss

tensor(0.0582, grad_fn=<NllLossBackward0>)

In [13]:
outputs.logits

tensor([[[ 15.1637,  -8.1877,   2.8143,  ..., -14.0755, -14.4777,  -8.2164],
         [ -7.4765, -16.4134,  -8.0899,  ..., -15.3485, -14.8985, -16.3759],
         [-10.7473, -15.0648,  -9.3695,  ..., -14.5629, -15.0044, -15.0701],
         ...,
         [-11.0623, -18.4554, -10.1851,  ..., -14.7522, -14.5882, -18.4171],
         [-11.6858, -15.5456,  -9.8657,  ..., -14.4394, -14.7645, -15.5139],
         [  0.9148,  -9.3476,  19.8629,  ..., -14.2594, -13.7868,  -9.3326]]],
       grad_fn=<AddBackward0>)

In [14]:
outputs.hidden_states

(tensor([[[ 0.0566, -0.0615, -0.1373,  ..., -0.2344,  0.1247, -0.0697],
          [ 0.0566, -0.0615, -0.1373,  ..., -0.2344,  0.1247, -0.0697],
          [-0.0420, -0.0421, -0.0509,  ..., -0.0582, -0.2253, -0.0768],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [-0.0420, -0.0421, -0.0509,  ..., -0.0582, -0.2253, -0.0768],
          [-0.0946, -0.0473, -0.0607,  ..., -0.0617,  0.0688, -0.1151]]],
        grad_fn=<MulBackward0>),
 tensor([[[ 0.6141, -1.1539,  5.5146,  ..., -1.6007,  5.0928,  4.2035],
          [ 0.6241, -1.1030,  5.5182,  ..., -1.5827,  5.0294,  4.1776],
          [ 0.7014, -1.7969,  6.0771,  ..., -0.9865,  4.4975,  4.6110],
          ...,
          [ 0.5393, -2.0274,  5.4777,  ..., -0.8895,  5.5866,  4.6522],
          [ 0.5729, -2.2042,  5.6824,  ..., -1.1087,  5.3204,  4.7404],
          [ 0.3686, -1.7037,  4.6428,  ..., -1.2950,  4.2940,  3.5389]]],
        grad_fn=<AddBackward0>),
 tensor([[[ 1.1186, -0.5455,  4.6038

# ESM2 from scratch

In [47]:
import torch
import esm

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()
#model.eval()  # disables dropout for deterministic results
model

ESM2(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0-32): 33 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=660, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((1280,), eps=1

In [48]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable parameter: {name}")

Trainable parameter: embed_tokens.weight
Trainable parameter: layers.0.self_attn.k_proj.weight
Trainable parameter: layers.0.self_attn.k_proj.bias
Trainable parameter: layers.0.self_attn.v_proj.weight
Trainable parameter: layers.0.self_attn.v_proj.bias
Trainable parameter: layers.0.self_attn.q_proj.weight
Trainable parameter: layers.0.self_attn.q_proj.bias
Trainable parameter: layers.0.self_attn.out_proj.weight
Trainable parameter: layers.0.self_attn.out_proj.bias
Trainable parameter: layers.0.self_attn_layer_norm.weight
Trainable parameter: layers.0.self_attn_layer_norm.bias
Trainable parameter: layers.0.fc1.weight
Trainable parameter: layers.0.fc1.bias
Trainable parameter: layers.0.fc2.weight
Trainable parameter: layers.0.fc2.bias
Trainable parameter: layers.0.final_layer_norm.weight
Trainable parameter: layers.0.final_layer_norm.bias
Trainable parameter: layers.1.self_attn.k_proj.weight
Trainable parameter: layers.1.self_attn.k_proj.bias
Trainable parameter: layers.1.self_attn.v_pro

In [49]:
del model.contact_head

In [50]:
def setup_model_for_tune(model):
    # freeze all layers but last one
    print("Freezing all layers but the last two...")
    num_layers = len(model.layers)
    n_trainable = 2
    trainable_blocks = [f"layers.{i}." for i in range(num_layers - n_trainable, num_layers)] 
    print(trainable_blocks)
    for name, param in model.named_parameters():
        param.requires_grad = any(block in name for block in trainable_blocks)

In [41]:
trainable_blocks = [f"layer.{i}." for i in range(33 - 2, 33)]
trainable_blocks

['layer.31.', 'layer.32.']

In [51]:
setup_model_for_tune(model)

Freezing all layers but the last two...
['layers.31.', 'layers.32.']


In [52]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable parameter: {name}")
      

Trainable parameter: layers.31.self_attn.k_proj.weight
Trainable parameter: layers.31.self_attn.k_proj.bias
Trainable parameter: layers.31.self_attn.v_proj.weight
Trainable parameter: layers.31.self_attn.v_proj.bias
Trainable parameter: layers.31.self_attn.q_proj.weight
Trainable parameter: layers.31.self_attn.q_proj.bias
Trainable parameter: layers.31.self_attn.out_proj.weight
Trainable parameter: layers.31.self_attn.out_proj.bias
Trainable parameter: layers.31.self_attn_layer_norm.weight
Trainable parameter: layers.31.self_attn_layer_norm.bias
Trainable parameter: layers.31.fc1.weight
Trainable parameter: layers.31.fc1.bias
Trainable parameter: layers.31.fc2.weight
Trainable parameter: layers.31.fc2.bias
Trainable parameter: layers.31.final_layer_norm.weight
Trainable parameter: layers.31.final_layer_norm.bias
Trainable parameter: layers.32.self_attn.k_proj.weight
Trainable parameter: layers.32.self_attn.k_proj.bias
Trainable parameter: layers.32.self_attn.v_proj.weight
Trainable par

In [4]:
data = [
    ("protein1", "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"),
    ("protein2", "KALTARQQEVFDLIRDHISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE"),
    ("protein2 with mask","KALTARQQEVFDLIRD<mask>ISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE"),
]
batch_labels, batch_strs, batch_tokens = batch_converter(data)
batch_tokens

tensor([[ 0,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  2],
        [ 0, 15,  5,  4, 11,  5, 10, 16, 16,  9,  7, 18, 13,  4, 12, 10, 13, 21,
         12,  8, 16, 11,  6, 20, 14, 14, 11, 10,  5,  9, 12,  5, 16, 10,  4,  6,
         18, 10,  8, 14, 17,  5,  5,  9,  9, 21,  4, 15,  5,  4,  5, 10, 15,  6,
          7, 12,  9, 12,  7,  8,  6,  5,  8, 10,  6, 12, 10,  4,  4, 16,  9,  9,
          2,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1],
        [ 0, 15,  5,  4, 11,  5, 10, 16, 16,  9,  7, 18, 13,  4, 12, 10, 1

In [5]:
out = model(batch_tokens, return_contacts=False)
logits = out['logits']
logits

tensor([[[ 14.8856,  -6.8767,   0.9738,  ..., -12.8237, -13.8168,  -6.9405],
         [ -7.4098, -11.3206,  -7.8506,  ..., -13.6347, -14.4332, -11.3238],
         [-10.0840, -12.0078,  -8.2523,  ..., -13.4121, -14.0281, -12.0006],
         ...,
         [-11.5679, -14.5259,  -9.5049,  ..., -13.5827, -14.0224, -14.5077],
         [-11.3315, -15.2855,  -9.7446,  ..., -14.0143, -14.4752, -15.2429],
         [  0.6718,  -6.2823,  19.2762,  ..., -13.1981, -12.9162,  -6.3313]],

        [[ 19.0375,  -5.0580,   0.5994,  ..., -14.3763, -14.9774,  -5.1247],
         [-12.0061, -16.3941, -12.5119,  ..., -15.4443, -15.6014, -16.3679],
         [-13.8951, -18.4356, -12.7754,  ..., -15.3292, -15.2819, -18.3107],
         ...,
         [  6.0818,  -2.7167,  27.1739,  ..., -14.6879, -13.7481,  -2.7864],
         [  5.9710,  -2.6150,  27.3550,  ..., -14.6267, -13.7358,  -2.6833],
         [  5.7897,  -2.6893,  26.9962,  ..., -14.6656, -13.8789,  -2.7770]],

        [[ 18.7514,  -5.2716,   0.6298,  ...

In [448]:
padding_token= alphabet.padding_idx
mask_token= alphabet.mask_idx
no_mask_tokens= [alphabet.cls_idx, alphabet.eos_idx]
n_tokens= len(alphabet.all_toks)
print(f"Padding token: {padding_token}, Mask token: {mask_token}, No mask tokens: {no_mask_tokens}, Vocabulary size: {n_tokens}")

Padding token: 1, Mask token: 32, No mask tokens: [0, 2], Vocabulary size: 33


In [449]:
MLM = MLM(
    padding_token=alphabet.padding_idx,
    mask_token=alphabet.mask_idx,
    no_mask_tokens=[alphabet.cls_idx, alphabet.eos_idx],
    n_tokens=len(alphabet.all_toks),
    masking_prob=0.15,
    randomize_prob=0.1,
    no_change_prob=0.1
)

In [450]:
batch_labels, batch_strs, batch_tokens = batch_converter(data)

x, y = MLM(batch_tokens)
x[0]

tensor([ 0,  5,  5,  5,  5,  5,  5,  5, 32,  5,  5,  5, 32,  5,  5,  5,  5,  5,
        32, 32,  5,  5,  5,  5,  5,  5,  5, 32,  5,  5,  5,  5,  5,  5,  5,  5,
         5,  5,  5, 32,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5, 29,  5,  5, 32,
         5,  5,  5,  5,  5, 32,  5,  5,  5,  5,  5,  5,  5,  5, 16,  5,  5,  5,
         5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
         5,  5,  5,  5,  5,  5,  5,  5,  5,  5, 32,  2])

In [451]:
y[0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 5, 1, 1, 1, 1, 1, 5, 5, 1, 1, 1, 1,
        1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 5, 1, 1, 5, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1,
        1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 5, 1])

In [452]:
count=0
for i in x[0]:
    if i !=5:
        count +=1

print(count)
count/len(x[0])

13


0.12745098039215685

In [454]:
output = model(x, return_contacts=False)
logits = output['logits']
logits

tensor([[[ 15.6507,  -6.8013,   2.0479,  ..., -12.8942, -13.6541,  -6.8510],
         [ -7.0689, -11.1746,  -7.2887,  ..., -13.6612, -14.4934, -11.1793],
         [-10.1540, -11.8307,  -7.9319,  ..., -13.3452, -13.9630, -11.8217],
         ...,
         [-10.8848, -13.9700,  -8.7343,  ..., -13.5893, -13.8973, -13.9372],
         [-10.4804, -17.4038,  -9.4925,  ..., -15.0059, -14.8071, -17.3493],
         [  1.5676,  -7.6133,  19.4869,  ..., -13.5665, -13.2796,  -7.6431]],

        [[ 17.7564,  -6.4637,   0.3475,  ..., -14.3206, -15.1642,  -6.5278],
         [ -9.8350, -15.4865, -11.8079,  ..., -15.6137, -15.2502, -15.4549],
         [-14.4077, -18.6080, -13.0556,  ..., -14.8976, -15.3059, -18.5193],
         ...,
         [  6.1821,  -2.8173,  26.9376,  ..., -14.6779, -13.9159,  -2.8768],
         [  5.9685,  -2.7357,  27.1306,  ..., -14.6466, -13.9056,  -2.7948],
         [  5.8408,  -2.8265,  26.9574,  ..., -14.7074, -14.1058,  -2.9101]],

        [[ 17.4094,  -5.7339,   0.4957,  ...

In [458]:
import torch.nn as nn
loss_fn = nn.CrossEntropyLoss(ignore_index=1)
loss_fn(logits.view(-1, logits.size(-1)), y.view(-1))

tensor(0.4765, grad_fn=<NllLossBackward0>)

In [463]:
logits.view(-1, logits.size(-1))#.shape

tensor([[ 15.6507,  -6.8013,   2.0479,  ..., -12.8942, -13.6541,  -6.8510],
        [ -7.0689, -11.1746,  -7.2887,  ..., -13.6612, -14.4934, -11.1793],
        [-10.1540, -11.8307,  -7.9319,  ..., -13.3452, -13.9630, -11.8217],
        ...,
        [  6.3847,  -2.7628,  27.5888,  ..., -14.7296, -13.8304,  -2.8285],
        [  6.2871,  -2.7040,  27.7447,  ..., -14.6706, -13.8144,  -2.7726],
        [  6.1231,  -2.7767,  27.4692,  ..., -14.6999, -13.9490,  -2.8671]],
       grad_fn=<ViewBackward0>)

In [464]:
y.view(-1)#.shape

tensor([ 1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  1,  5,  1,  1,  1,  1,  1,
         5,  5,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  5,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  5,
         1,  1,  1,  1,  1,  5,  1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  1,
         1,  1,  5,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  5,  1,  1,  1,  1,  1,  1,  1,
         1,  1, 16,  9,  7,  1,  1,  4,  1,  1,  1,  1, 12,  1,  1,  1,  1,  1,
         1,  1,  1, 10,  1,  1,  1,  5,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  9,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  9, 12,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  4,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1, 

## Embeddings from ESM-2 3B tuned

In [3]:
import os
import torch
import argparse
from Bio import SeqIO
from tqdm import tqdm

from transformers import EsmForMaskedLM, AutoTokenizer
from peft import PeftModel, PeftConfig

In [5]:
 # Load base model
base_model = EsmForMaskedLM.from_pretrained('facebook/esm2_t36_3B_UR50D')
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t36_3B_UR50D')
# Load the PEFT adapter
model = PeftModel.from_pretrained(base_model, '../checkpoints/rsawhney_esm2_3B/')
model.eval()  
model.to('cuda') 

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 18.52it/s]


PeftModel(
  (base_model): LoraModel(
    (model): EsmForMaskedLM(
      (esm): EsmModel(
        (embeddings): EsmEmbeddings(
          (word_embeddings): Embedding(33, 2560, padding_idx=1)
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (encoder): EsmEncoder(
          (layer): ModuleList(
            (0-35): 36 x EsmLayer(
              (attention): EsmAttention(
                (self): EsmSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=2560, out_features=2560, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Identity()
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2560, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=2560, bias=False)
                    )
                    (lora

In [6]:
batch_seqs = ['AAAAA', 'MASDFGH']
tokens = tokenizer(batch_seqs, return_tensors="pt", padding=True)
output = model(**tokens.to('cuda'), output_hidden_states=True)
output 

MaskedLMOutput(loss=None, logits=tensor([[[ 1.7357e+01, -5.5867e+00, -2.8229e-01, -5.4016e+00, -2.2212e+00,
           5.4093e-01, -2.2531e-01, -1.1676e+00, -7.2219e-01, -1.0030e+00,
          -2.1634e+00, -1.4692e+00, -2.7173e+00, -1.1345e+00, -2.1959e+00,
          -1.5757e+00, -3.7025e+00, -2.6211e+00, -3.7678e+00, -1.8661e+00,
          -2.6211e-01, -4.3865e+00, -7.1192e+00, -3.4298e+00, -2.7694e+00,
          -1.5582e+01, -5.9198e+00, -1.1538e+01, -1.6932e+01, -1.5178e+01,
          -1.5242e+01, -1.2304e+01, -5.5745e+00],
         [-1.0157e+01, -1.6276e+01, -1.1632e+01, -1.6182e+01, -1.7239e+00,
           4.9158e+00, -5.8565e-01, -1.2123e+00, -1.2727e+00, -1.8293e+00,
          -9.8121e-01, -1.6891e+00, -2.9671e+00, -2.0458e+00, -8.3462e-01,
          -2.2132e+00, -2.3462e+00, -3.4831e+00, -2.5899e+00, -2.7738e+00,
           1.9119e+00, -3.5153e+00, -2.7494e+00, -1.2660e+00, -2.1426e+00,
          -1.2697e+01, -1.1563e+01, -1.1663e+01, -1.5873e+01, -1.4805e+01,
          -1.4702

In [14]:
output['hidden_states'][0]

tensor([[[-0.0479,  0.0190,  0.0211,  ..., -0.1953,  0.0290,  0.0575],
         [-0.0071, -0.0569, -0.0036,  ..., -0.2342, -0.0971, -0.0631],
         [-0.0071, -0.0569, -0.0036,  ..., -0.2342, -0.0971, -0.0631],
         ...,
         [-0.0229,  0.1117,  0.1141,  ..., -0.0575,  0.0542,  0.0037],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

        [[-0.0479,  0.0190,  0.0211,  ..., -0.1953,  0.0290,  0.0575],
         [-0.0138,  0.1366, -0.0114,  ...,  0.0257,  0.0444, -0.1253],
         [-0.0071, -0.0569, -0.0036,  ..., -0.2342, -0.0971, -0.0631],
         ...,
         [-0.0044,  0.0064, -0.0423,  ...,  0.0187, -0.0709, -0.0671],
         [-0.0034, -0.0126,  0.0536,  ...,  0.0098, -0.0338,  0.2496],
         [-0.0229,  0.1117,  0.1141,  ..., -0.0575,  0.0542,  0.0037]]],
       device='cuda:0')

In [15]:
output['hidden_states'][-1]

tensor([[[ 0.0529, -0.0391, -0.0619,  ...,  0.0342,  0.0054, -0.1149],
         [-0.0087, -0.1791, -0.0954,  ...,  0.0326, -0.1875, -0.0999],
         [-0.0650, -0.0358, -0.0574,  ..., -0.0536, -0.1895, -0.0547],
         ...,
         [ 0.0752,  0.0130, -0.0432,  ..., -0.0279, -0.0629, -0.1003],
         [ 0.0166, -0.0418, -0.0528,  ..., -0.0509, -0.1071, -0.1582],
         [ 0.0614, -0.0015, -0.0462,  ..., -0.0747, -0.1362, -0.1288]],

        [[ 0.0283, -0.0409, -0.0597,  ...,  0.0616,  0.0187, -0.1001],
         [ 0.0482, -0.2345,  0.0422,  ..., -0.0722, -0.2990, -0.0204],
         [ 0.0142, -0.0838,  0.0498,  ..., -0.0864, -0.3162, -0.0321],
         ...,
         [ 0.0696, -0.1350,  0.2150,  ..., -0.0237, -0.2979,  0.0046],
         [-0.2188,  0.0269,  0.0615,  ..., -0.1030, -0.3772,  0.0346],
         [ 0.0093, -0.0029,  0.0462,  ..., -0.0171, -0.1411, -0.1486]]],
       device='cuda:0')

In [19]:
output['hidden_states'][36].shape

torch.Size([2, 9, 2560])

In [ ]:
embeddings = output['hidden_states'][0]

# Span masking

In [1]:
import numpy as np

def whole_span_mask(seq_length, mask_prob=0.15, mean_span_len=3):
    """Return a mask array for span masking."""
    n_to_mask = int(seq_length * mask_prob)
    mask = np.zeros(seq_length, dtype=bool)
    masked = 0
    
    while masked < n_to_mask:
        span_len = np.random.geometric(1.0/mean_span_len)
        start = np.random.randint(0, seq_length - span_len + 1)
        end = min(start + span_len, seq_length)
        
        if not mask[start:end].any():  # avoid overlapping spans
            mask[start:end] = True
            masked += (end - start)
    
    return mask


In [2]:
whole_span_mask(100)

array([ True, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False,  True, False, False, False, False, False,  True,  True,
        True,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True,  True, False, False,
        True,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True,  True,  True, False, False,
       False, False, False, False, False, False, False, False, False,
       False])

In [ ]:
def _make_span_mask(seq_len: int, masking_prob: float, mean_span_len: int = 3, device="cpu"):
    """
    Create a boolean mask for span masking.
    About `masking_prob` fraction of positions are True.
    """
    n_to_mask = int(seq_len * masking_prob)
    mask = torch.zeros(seq_len, dtype=torch.bool, device=device)
    masked = 0

    while masked < n_to_mask:
        # sample span length from geometric
        span_len = np.random.geometric(1.0 / mean_span_len)
        start = np.random.randint(0, seq_len - span_len + 1)
        end = min(start + span_len, seq_len)

        # avoid overlaps
        if not mask[start:end].any():
            mask[start:end] = True
            masked += (end - start)

    return mask


class MLM:
    def __init__(self, *,
                 padding_token: int, mask_token: int, no_mask_tokens: List[int], n_tokens: int,
                 masking_prob: float = 0.15, randomize_prob: float = 0.1, no_change_prob: float = 0.1,
                 mean_span_len: int = 3):
        self.n_tokens = n_tokens
        self.no_change_prob = no_change_prob
        self.randomize_prob = randomize_prob
        self.masking_prob = masking_prob
        self.no_mask_tokens = no_mask_tokens + [padding_token, mask_token]
        self.padding_token = padding_token
        self.mask_token = mask_token
        self.mean_span_len = mean_span_len

    def __call__(self, x: torch.Tensor):
        """
        x: [seq_len, batch_size]
        """
        seq_len, batch_size = x.shape
        full_mask = torch.zeros_like(x, dtype=torch.bool)

        # build span masks independently for each sequence
        for b in range(batch_size):
            mask_b = _make_span_mask(seq_len, self.masking_prob, self.mean_span_len, device=x.device)
            # forbid masking special tokens
            for t in self.no_mask_tokens:
                mask_b &= (x[:, b] != t)
            full_mask[:, b] = mask_b

        # same logic as before
        unchanged = full_mask & (torch.rand(x.shape, device=x.device) < self.no_change_prob)
        random_token_mask = full_mask & (torch.rand(x.shape, device=x.device) < self.randomize_prob)

        random_token_idx = torch.nonzero(random_token_mask, as_tuple=True)
        random_tokens = torch.randint(0, self.n_tokens, (len(random_token_idx[0]),), device=x.device)

        mask = full_mask & ~random_token_mask & ~unchanged

        y = x.clone()
        x.masked_fill_(mask, self.mask_token)
        x[random_token_idx] = random_tokens
        y.masked_fill_(~full_mask, self.padding_token)

        return x, y


# Learning rate

In [11]:
lr = 1e-4
print(f'Ori lr: {lr}')
for i in range(10):
    lr = lr * 0.5
    print(f'New lr: {lr}')


Ori lr: 0.0001
New lr: 5e-05
New lr: 2.5e-05
New lr: 1.25e-05
New lr: 6.25e-06
New lr: 3.125e-06
New lr: 1.5625e-06
New lr: 7.8125e-07
New lr: 3.90625e-07
New lr: 1.953125e-07
New lr: 9.765625e-08


In [10]:
4e-4 * 0.01

4.000000000000001e-06

In [5]:
4e-4 * 0.1

4e-05

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Dummy model
model = nn.Linear(10, 1)
optimizer = optim.Adam(model.parameters(), lr=4e-4)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=0, cooldown=0)

# Dummy training loop
losses = [0.1, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09, 0.09]  # example losses
for epoch, loss_val in enumerate(losses, 1):
    # pretend we did a forward/backward pass here
    optimizer.step()  # normally after loss.backward()
    
    # step the scheduler with validation loss
    scheduler.step(loss_val)
    
    print(f"Epoch {epoch}, LR: {optimizer.param_groups[0]['lr']}")
